# Random Forest

Standard tabular ML workflow for LD50 regression.


In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
from sklearn.ensemble import RandomForestRegressor

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.modeling import artifact_paths, load_tabular_arrays, save_ml_run


In [2]:
MODEL_NAME = "random_forest"
BASE_DIR = Path.cwd()
RANDOM_STATE = 42

X_train, X_valid, X_test, y_train, y_valid, y_test, feature_names = load_tabular_arrays('../../data/feature_selection')
X_train = X_train.astype(np.float32)
X_valid = X_valid.astype(np.float32)
X_test = X_test.astype(np.float32)

paths = artifact_paths(
    BASE_DIR,
    MODEL_NAME,
    model_suffix=".pkl",
    include_importance=True,
)


In [3]:
# Hypertuning
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions={
        "n_estimators": [500, 700, 800, 1000],
        "max_depth": [None, 10, 20, 40],
        "min_samples_split": [1, 2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 1.0],
    },
    n_iter=12,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=False,
)
search.fit(X_valid, y_valid)

best_params = search.best_params_
print(f"[Random Forest hypertuning] Best params: {best_params} | validation RMSE: {-search.best_score_:.4f}")

[Random Forest hypertuning] Best params: {'n_estimators': 800, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20} | validation RMSE: 0.7958


c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
15 fits failed out of a total of 36.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\sklearn\base.py", line 1358, in wrapper
    estimator._validate_params()
  File "c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\sklearn\base.py", line 471, in _validate_pa

In [4]:
try:
    from cuml.ensemble import RandomForestRegressor as CumlRandomForest

    model = CumlRandomForest(
        **best_params,
        random_state=RANDOM_STATE,
    )
except Exception:
    warnings.warn("cuML RandomForestRegressor not available; using sklearn on CPU.")
    model = RandomForestRegressor(
        **best_params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

model.fit(X_train, y_train)
predictions = model.predict(X_test)

C:\Users\Tommaso\AppData\Local\Temp\ipykernel_38944\3916803871.py:9: UserWarning: cuML RandomForestRegressor not available; using sklearn on CPU.
  warnings.warn("cuML RandomForestRegressor not available; using sklearn on CPU.")


In [5]:
metrics = save_ml_run("Random Forest", model, paths, X_test, y_test, predictions, feature_names)

[Random Forest] RMSE: 0.8915 | MAE: 0.6402 | R2: 0.2908
